In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("../data/vanilla_convertibles_data_enhanced.csv")

In [ ]:
df = df.dropna(subset=["price_normalized"])

X = df.drop(columns=["price_convertible", "price_normalized", "redemption"])
y = df[["price_normalized"]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42)


In [ ]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing._data import BOUNDS_THRESHOLD
from scipy.stats import norm

scaler_X = QuantileTransformer(output_distribution="normal", random_state=42)
scaler_y = QuantileTransformer(output_distribution="normal", random_state=42)

X_train = scaler_X.fit_transform(X_train)
X_val = scaler_X.transform(X_val)
X_test = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train)
y_val = scaler_y.transform(y_val)

_lower = norm.ppf(BOUNDS_THRESHOLD)
_upper = norm.ppf(1 - BOUNDS_THRESHOLD)
X_train = np.clip(X_train, _lower, _upper)
X_val = np.clip(X_val, _lower, _upper)
X_test = np.clip(X_test, _lower, _upper)
y_train = np.clip(y_train, _lower, _upper)
y_val = np.clip(y_val, _lower, _upper)

In [ ]:
import matplotlib.pyplot as plt

n_estimators_range = [10, 25, 50, 100, 150, 200, 300, 400, 500, 750, 1000, 1250, 1500]
val_rmses = []

for n in n_estimators_range:
    gbr_sweep = GradientBoostingRegressor(n_estimators=n, random_state=42)
    gbr_sweep.fit(X_train, y_train)
    y_pred = gbr_sweep.predict(X_val)
    val_rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))
    print(f"n_estimators={n:>5d} | Val RMSE: {val_rmses[-1]:.6f}")

plt.figure(figsize=(10, 5))
plt.plot(n_estimators_range, val_rmses, marker="o")
plt.xlabel("n_estimators")
plt.ylabel("Validation RMSE")
plt.title("Gradient Boosting — n_estimators Elbow Analysis")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
gbr = GradientBoostingRegressor(random_state=42)

param_grid_gbr = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.8, 1.0],
    "max_features": [None, "sqrt", "log2"]
}

grid_search_gbr = GridSearchCV(
    estimator=gbr,
    param_grid=param_grid_gbr,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

grid_search_gbr.fit(X_train, y_train)

print("Best GBR parameters:", grid_search_gbr.best_params_)
print("Best GBR CV RMSE:", -grid_search_gbr.best_score_)

In [ ]:
best_gbr = grid_search_gbr.best_estimator_

y_pred_val = best_gbr.predict(X_val)
y_pred_test = best_gbr.predict(X_test)

print("--- Validation ---")
print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred_val)))
print("MAE:", mean_absolute_error(y_val, y_pred_val))
print("R^2:", r2_score(y_val, y_pred_val))

print("\n--- Test ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
print("MAE:", mean_absolute_error(y_test, y_pred_test))
print("R^2:", r2_score(y_test, y_pred_test))